# Summer Storm Langlois Experiment HI

In [24]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os 
from scipy.optimize import curve_fit
from scipy.integrate import quad

experiment_directory = 'pulse_experiments'
experiments = {}
for filename in os.listdir(experiment_directory):
    # check if the file is a CSV file
    if filename.endswith('.csv'):
        file_path = os.path.join(experiment_directory, filename) # construct the full file path
        df = pd.read_csv(file_path)                         # read the CSV file into a data frame
        df = df.dropna(subset=['Date_Time'])                # drop rows where 'Date/Time' is NaN  
        df['Date_Time'] = pd.to_datetime(df['Date_Time'])   # convert to datetime format
        df = df.set_index('Date_Time')                      # set date time as the index 
        df = df.dropna(how='all', axis=1)                   # drop columns where all values are NaN
        df = df.loc[:, ~df.columns.astype(str).str.startswith('Unnamed')]  # drop stray unnamed columns
        key = filename[:-4]                                 # remove the '.csv' from the filename to use as the dictionary key
        experiments[key] = df                                      # store the data frame in the dictionary

# calculate a shear stress column for each experiment
for experiment_name, experiment_df in experiments.items():
    # calculate shear stress using the formula: shear_stress = density * gravity * water_depth * slope
    density = 1000  # kg/m^3
    gravity = 9.81  # m/s^2
    slope = 0.098
    water_depth = experiment_df['Depth']  # in cm
    shear_stress = density * gravity * water_depth/1000 * slope
    experiments[experiment_name]['shear_stress'] = shear_stress

In [25]:
experiments['Experiment1']

,Lab ID,Depth,SSC (uL/L),SSC (mg/L),DOC (mg/L),POC (mg/L),shear_stress
Date_Time,,,,,,,
2022-08-02 11:57:09,1,9.275,97.69,167.044168,2.640,10.974524,8.916800
2022-08-02 11:57:21,2,22.750,347.61,145.078620,2.384,12.330497,21.871395
2022-08-02 11:57:32,3,19.250,191.85,76.137885,2.284,NaN,18.506565
2022-08-02 11:57:42,4,16.100,176.97,47.967685,2.423,4.967149,15.478218
2022-08-02 11:57:53,5,13.884,127.27,67.326074,2.488,8.890353,13.347800
2022-08-02 11:58:04,6,12.775,83.04,61.268422,2.473,NaN,12.281630
2022-08-02 11:58:14,7,12.134,68.69,40.782355,2.314,NaN,11.665385
2022-08-02 11:58:25,8,11.959,62.04,43.350237,2.402,NaN,11.497143
2022-08-02 11:58:34,9,11.959,61.07,37.341299,2.278,NaN,11.497143


Hysteresis index calculation functions

In [26]:
## regression equations
# linear
def linear_func(Q, a, b):
    return a * Q + b
# logarithmic
def log_func(Q, a, b):
    return a * np.log(Q) + b
# exponential
def exp_func(Q, a, b):
    return a * np.exp(b * Q)

# split hydrograph into rising and falling limbs based on peak flow
def split_hydrograph(df, q_col):
    # if df empty or q_col has no valid values, return empty limbs
    if df.empty or df[q_col].dropna().empty:
        return df.copy(), df.copy()
    peak_time = df[q_col].idxmax()
    rising = df.loc[:peak_time].copy()
    falling = df.loc[peak_time:].copy()
    return rising, falling

# fit curves and calculate R²
def fit_best_curve(x, y):
    candidate_functions = {
        'linear': (linear_func, [1, 1]),
        'log': (log_func, [1, 1]),
        'exponential': (exp_func, [1, -0.01])
    }
    x = pd.to_numeric(x, errors="coerce")
    y = pd.to_numeric(y, errors="coerce")
    mask = np.isfinite(x) & np.isfinite(y)
    x = np.asarray(x[mask])
    y = np.asarray(y[mask])

    # minimum points
    if len(x) < 2:
        return None
    best_r2 = -np.inf
    best_result = None
    # try each function and keep the one with the best r2
    for func_name, (func, p0) in candidate_functions.items():
        try:
            # avoid invalid log fits
            if func_name == 'log' and np.any(x <= 0):
                continue
            popt, _ = curve_fit(func, x, y, p0=p0, maxfev=20000) # fit curve
            y_pred = func(x, *popt) # predicted values
            # residuals
            ss_res = np.sum((y - y_pred) ** 2)
            ss_tot = np.sum((y - np.mean(y)) ** 2)
            # avoid divide-by-zero
            if np.isclose(ss_tot, 0):
                r2 = np.nan
            else:
                r2 = 1 - (ss_res / ss_tot)
            # keep best fit
            if np.isfinite(r2) and r2 > best_r2:
                best_r2 = r2
                best_result = {
                    'function_name': func_name,
                    'function': func,
                    'params': popt,
                    'r2': r2
                }
        except Exception:
            continue
    return best_result

# Langlois 2025 H calculation
def compute_langlois_H(event_df, tau_col, constituent_col, r2_threshold=0.50, storm_name=None):
    # data cleanup
    df = event_df[[tau_col, constituent_col]].dropna()
    if df.empty or df[tau_col].dropna().empty or df[constituent_col].dropna().empty:
        print(f"No valid {tau_col} or {constituent_col} for {storm_name or 'unknown'}; skipping")
        return None

    rising, falling = split_hydrograph(df, tau_col)
    if rising.empty or falling.empty:
        print(f"No rising or falling limb for {storm_name or 'unknown'}; skipping")
        return None

    # shear stress overlap range
    tau_min = max(rising[tau_col].min(), falling[tau_col].min())
    tau_max = min(rising[tau_col].max(), falling[tau_col].max())

    # fit rising limb
    rise_fit = fit_best_curve(rising[tau_col].values, rising[constituent_col].values)
    if rise_fit is None:
        print(f"Could not fit rising limb in " f"{storm_name or 'unknown'} for {constituent_col}")
        return None
    # fit falling limb
    fall_fit = fit_best_curve(falling[tau_col].values, falling[constituent_col].values)
    if fall_fit is None:
        print(f"Could not fit falling limb in " f"{storm_name or 'unknown'} for {constituent_col}")
        return None

    # check fit quality with r2 threshold
    if (rise_fit['r2'] < r2_threshold) or (fall_fit['r2'] < r2_threshold):
            label = storm_name if storm_name is not None else "unknown storm"
            print(f"Poor fit for rising (R²={rise_fit['r2']:.2f}) or falling (R²={fall_fit['r2']:.2f}) limb in {label} for {constituent_col}")
    # integrated areas
    rise_area, _ = quad(lambda q: rise_fit["function"](q, *rise_fit["params"]), tau_min, tau_max)
    fall_area, _ = quad(lambda q: fall_fit["function"](q, *fall_fit["params"]), tau_min, tau_max)
    # hysteresis index
    H = rise_area / fall_area

    return {
        # hysteresis
        'H': H,
        # rising limb
        'rise_r2': rise_fit['r2'],
        'rise_function': rise_fit['function'],
        'rise_params': rise_fit['params'],
        'rise_area': rise_area,
        # falling limb
        'fall_r2': fall_fit['r2'],
        'fall_function': fall_fit['function'],
        'fall_params': fall_fit['params'],
        'fall_area': fall_area,
        # overlap range
        'tau_min': tau_min,
        'tau_max': tau_max,
        # point counts
        'n_rising': len(rising),
        'n_falling': len(falling)
    }

Calculate H for all events

In [27]:
all_results = []

for experiment_name, experiment_df in experiments.items():
    for constituent in ["SSC (mg/L)", "DOC (mg/L)", "POC (mg/L)"]:
        if constituent not in experiment_df.columns:
            continue

        result = compute_langlois_H(
            experiment_df,
            tau_col="Depth",
            constituent_col=constituent,
            storm_name=experiment_name)

        if result is not None:
            result["experiment"] = experiment_name
            result["constituent"] = constituent
            all_results.append(result)

all_results = pd.DataFrame(all_results)
all_results.to_csv('HI_calculations/langlois_experiment_hysteresis.csv', index=False)

Poor fit for rising (R²=1.00) or falling (R²=0.07) limb in Experiment1 for DOC (mg/L)
Poor fit for rising (R²=0.79) or falling (R²=0.08) limb in Experiment2 for DOC (mg/L)
Poor fit for rising (R²=1.00) or falling (R²=0.31) limb in Experiment3 for DOC (mg/L)
Poor fit for rising (R²=0.16) or falling (R²=0.92) limb in Experiment4 for SSC (mg/L)
Poor fit for rising (R²=0.11) or falling (R²=0.96) limb in Experiment4 for POC (mg/L)
Poor fit for rising (R²=0.01) or falling (R²=0.72) limb in Experiment5 for SSC (mg/L)
Poor fit for rising (R²=0.46) or falling (R²=0.58) limb in Experiment5 for DOC (mg/L)
Poor fit for rising (R²=0.10) or falling (R²=0.95) limb in Experiment5 for POC (mg/L)
Poor fit for rising (R²=0.39) or falling (R²=0.02) limb in Experiment6 for DOC (mg/L)
Poor fit for rising (R²=0.19) or falling (R²=1.00) limb in Experiment6 for POC (mg/L)
Poor fit for rising (R²=0.49) or falling (R²=0.17) limb in Experiment8 for DOC (mg/L)


C:\Users\nicol\AppData\Local\Temp\ipykernel_25116\1718751059.py:46: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(func, x, y, p0=p0, maxfev=20000) # fit curve


### Plots

In [28]:
def plot_langlois_hysteresis(event_df, tau_col, constituent_col, r2_threshold=0.5, storm_name=None, 
                            out_dir='plots', save=True, show=False):

    df = event_df[[tau_col, constituent_col]].dropna()
    if df.empty:
        return None
    rising, falling = split_hydrograph(df, tau_col)
    if rising.empty or falling.empty:
        return None
    
    # reuse the same fitting logic as the H calculation
    result = compute_langlois_H(
        event_df,
        tau_col=tau_col,
        constituent_col=constituent_col,
        r2_threshold=r2_threshold,
        storm_name=storm_name,
    )
    if result is None:
        return None

    tau_min = result["tau_min"]
    tau_max = result["tau_max"]
    if not np.isfinite(tau_min) or not np.isfinite(tau_max) or tau_min >= tau_max:
        return None

    tau_fit = np.linspace(tau_min, tau_max, 200)

    rise_params = result["rise_params"]
    fall_params = result["fall_params"]
    rise_r2 = result["rise_r2"]
    fall_r2 = result["fall_r2"]
    H = result["H"]

    rise_fit = result["rise_function"](tau_fit, *result["rise_params"])
    fall_fit = result["fall_function"](tau_fit, *result["fall_params"])

    # PLOT 
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    # time series
    ax = axes[0]
    ax.plot(df.index, df[tau_col], color="tab:blue", linewidth=1.5, label=tau_col)
    ax.set_ylabel(tau_col, color="tab:blue")
    ax.tick_params(axis="y", labelcolor="tab:blue")
    ax.xaxis.set_major_locator(plt.MaxNLocator(8))
    ax2 = ax.twinx()
    ax2.plot(df.index, df[constituent_col], color="tab:red", linewidth=1.5, label=constituent_col)
    ax2.set_ylabel(constituent_col, color="tab:red")
    ax2.tick_params(axis="y", labelcolor="tab:red")
    ax.set_title("Event time series")

    # hysteresis loop
    ax = axes[1]
    ax.scatter(rising[tau_col], rising[constituent_col], label='Rising limb', color='tab:orange')
    ax.scatter(falling[tau_col], falling[constituent_col], label='Falling limb', color='tab:green')
    ax.plot(tau_fit, rise_fit, linewidth=2, label=f'Rising fit (R²={rise_r2:.2f})', color='tab:orange')
    ax.plot(tau_fit, fall_fit, linewidth=2, label=f'Falling fit (R²={fall_r2:.2f})', color='tab:green')

    ax.set_xlabel(tau_col)
    ax.set_ylabel(constituent_col)
    ax.set_title(f'H = {H:.2f}')
    ax.legend()

    # add a main title for the whole figure
    main_title = f"{storm_name} - {constituent_col} Hysteresis" if storm_name else f"{constituent_col} Hysteresis"
    plt.suptitle(main_title, fontsize=15)
    plt.tight_layout()

    if save:
        os.makedirs(out_dir, exist_ok=True)
        safe_name = f"{storm_name}_{constituent_col}_langlois.png".replace(" ", "_").replace("/", "_")
        fig.savefig(os.path.join(out_dir, safe_name), dpi=300, bbox_inches="tight")

    if show:
        plt.show()
    plt.close(fig)
    return result

In [29]:
all_results = []

for experiment_name, experiment_df in experiments.items():
    for constituent in ["SSC (mg/L)", "DOC (mg/L)", "POC (mg/L)"]:
        if constituent not in experiment_df.columns:
            continue

        plot_langlois_hysteresis(
            experiment_df,
            tau_col="Depth",
            constituent_col=constituent,
            storm_name=experiment_name,
            out_dir='plots/langlois',)

C:\Users\nicol\AppData\Local\Temp\ipykernel_25116\1718751059.py:46: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(func, x, y, p0=p0, maxfev=20000) # fit curve


Poor fit for rising (R²=1.00) or falling (R²=0.07) limb in Experiment1 for DOC (mg/L)
Poor fit for rising (R²=0.79) or falling (R²=0.08) limb in Experiment2 for DOC (mg/L)
Poor fit for rising (R²=1.00) or falling (R²=0.31) limb in Experiment3 for DOC (mg/L)
Poor fit for rising (R²=0.16) or falling (R²=0.92) limb in Experiment4 for SSC (mg/L)
Poor fit for rising (R²=0.11) or falling (R²=0.96) limb in Experiment4 for POC (mg/L)
Poor fit for rising (R²=0.01) or falling (R²=0.72) limb in Experiment5 for SSC (mg/L)
Poor fit for rising (R²=0.46) or falling (R²=0.58) limb in Experiment5 for DOC (mg/L)
Poor fit for rising (R²=0.10) or falling (R²=0.95) limb in Experiment5 for POC (mg/L)
Poor fit for rising (R²=0.39) or falling (R²=0.02) limb in Experiment6 for DOC (mg/L)
Poor fit for rising (R²=0.19) or falling (R²=1.00) limb in Experiment6 for POC (mg/L)
Poor fit for rising (R²=0.49) or falling (R²=0.17) limb in Experiment8 for DOC (mg/L)
